In [1]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from pyvis.network import Network
from langchain.chat_models import init_chat_model
import pandas as pd
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate
from typing import Tuple, Optional
import re

from dotenv import load_dotenv
import os
import asyncio

os.environ["DEEPSEEK_API_KEY"] = "sk-ae6d423f13b7403fa8ba16ddb4acc297"

llm_deepseek=init_chat_model(
    model='deepseek:deepseek-chat',
    temperature=1.3)

# Load the .env file
load_dotenv()
api_key = os.getenv("DEEPSEEK_API_KEY")

In [20]:
llm_qwen=init_chat_model(
    model='deepseek:deepseek-chat',
    temperature=1.3)

# Load the .env file
load_dotenv()
api_key = os.getenv("DEEPSEEK_API_KEY")

In [2]:
def get_question_text(row: pd.Series) -> str:
    """将材料和提问组合成完整的题干"""
    material = str(row['材料']) if pd.notna(row['材料']) else ''
    question = str(row['提问']) if pd.notna(row['提问']) else ''
    # 如果材料非空且不以标点结尾，加个句号使语义完整
    if material and not material[-1] in '。？！；;':
        material += '。'
    return material + question

In [3]:
def get_options_text(row: pd.Series) -> str:
    """将四个选项拼接为文本"""
    opts = []
    for i in range(1, 5):
        opt = row.get(f'选项{i}', '')
        if pd.notna(opt) and str(opt).strip():
            opts.append(f"{chr(64+i)}. {opt}")
    return '\n'.join(opts)

In [10]:
def parse_llm_response(response: str) -> Tuple[Optional[str], Optional[str]]:
    """
    解析模型返回的字符串，提取难度和认知层级。
    期望格式示例：
        难度：中等
        认知层级：理解
    """
    difficulty, cognitive = None, None
    
    # 匹配难度：后面跟两个字（简单、中等、困难）
    diff_match = re.search(r'难度[:：]\s*([\u4e00-\u9fa5]{2})', response)
    if diff_match:
        difficulty = diff_match.group(1)
    
    # 匹配认知层级：后面跟两个字（记忆、理解、应用、分析、评价、创造）
    cog_match = re.search(r'认知层级[:：]\s*([\u4e00-\u9fa5]{2})', response)
    if cog_match:
        cognitive = cog_match.group(1)
    
    return difficulty, cognitive

In [11]:
def labeling(input_excel: str, output_excel: str):
    """
    主函数：读取Excel，为每一题调用LLM打标签，输出新Excel。
    """
    # 1. 读取Excel，所有列作为字符串读入，避免科学计数法等干扰
    df = pd.read_excel(input_excel, dtype=str)
    
    # 2. 准备存储结果的列表
    difficulties = []
    cognitive_levels = []
    
    # 3. 系统提示词
    system_prompt = """
        你是一名专业的高考地理命题专家，擅长根据试题的题干和选项判断其难度和考察的认知层级。

        一、难度系数判定标准（三级）
        难度分为三级：简单、中等、困难。
        （1）简单
        特征：基础事实识记与典型景观辨识
        判定依据：
        单一知识点考查，如直接判定地貌基本类型或主要外力名称
        景观特征描述直观，材料包含明确的核心关键词
        选项干扰性弱，直接对应教材中的基础定义或术语
        无需复杂的空间逻辑推理或因果链条分析
        示例：识别新月形沙丘的成因、喀斯特地貌的岩石基础、风蚀蘑菇的动力来源
        （2）中等
        特征：地貌成因理解与概念规律辨析
        判定依据：
        考查地貌形成的动态演变过程或先后发育顺序
        需准确区分相似概念或同一外力的不同表现形式
        涉及简单的地理图表解析，需从示意图或剖面图中提取有效信息
        存在一定干扰项，要求对地理原理有准确的辨析能力
        示例：分析河流凹岸与凸岸的性质差异、冲积扇沉积物粒径的分选规律、判断特定地貌在我国的典型分布区域
        （3）困难
        特征：综合思维应用与新情境逻辑推理
        判定依据：
        多要素关联考查，涉及气候、地质、水文及人类活动对地貌的综合影响
        考查非典型区域的特殊成因地貌或较冷门的专业术语
        材料情境陌生或复杂，需要将地理原理迁移至新情境中解决实际问题
        逻辑因果链条长，选项具有较强的学术诱惑力和干扰性
        示例：嵌入式蛇曲的形成环境分析、地坑院建筑与黄土物理性质的关系推导、特定工程选址与微地貌排水条件的综合评估

        二、认知维度判定标准（布鲁姆六级）
        共六级：记忆、理解、应用、分析、评价、创造。
        1. 记忆（Remembering）
        定义：回忆、识别已学的具体信息
        关键动词：是什么、定义、列举、识别、命名、复述
        判定要点：题目答案可直接从记忆中提取，无需理解转化
        无人机领域示例：
        UAV的英文全称是什么？
        列举无人机的基本组成部分
        超近程无人机的活动半径范围
        2. 理解（Understanding）
        定义：解释信息含义，理解概念间关系
        关键动词：解释、说明、概括、区分、比较、归纳、转换
        判定要点：需要用自己的话解释或区分概念差异
        无人机领域示例：
        解释多旋翼与固定翼的主要区别
        说明GPS定位系统在无人机中的作用
        比较不同动力系统的优缺点
        3. 应用（Applying）
        定义：在新的具体情境中使用已学知识
        关键动词：应用、使用、执行、实施、解决、计算、操作
        判定要点：给定具体场景，选择或执行正确的方法
        无人机领域示例：
        根据任务需求选择合适的无人机类型
        计算特定条件下的续航时间
        为农田作业设计飞行参数
        4. 分析（Analyzing）
        定义：分解信息，理解各部分关系及组织结构
        关键动词：分析、对比、检查、区别、推断、诊断、识别原因
        判定要点：需要识别因果关系、找出关键要素或推断原因
        无人机领域示例：
        分析导致飞行事故的可能原因
        诊断系统故障的类型及影响
        推断不同气象条件对飞行的影响
        5. 评价（Evaluating）
        定义：基于标准和准则做出价值判断
        关键动词：评估、评价、判断、检验、批判、论证、辩护、选择优劣
        判定要点：需要根据标准评判方案优劣或合理性
        无人机领域示例：
        评估某飞行方案的安全性和可行性
        判断操作流程是否符合法规要求
        论证设备选型的合理性
        6. 创造（Creating）
        定义：整合各要素形成新的结构或方案
        关键动词：设计、规划、创建、开发、构建、组合、重组
        判定要点：需要综合多种知识创建新方案
        无人机领域示例：
        设计完整的飞行任务方案
        规划复杂地形的航线路径
        开发针对特定场景的应用解决方案

        请严格按以下格式输出，不要输出额外解释：
        难度：简单/中等/困难
        认知层级：记忆/理解/应用/分析/评价/创造
        """

    # 4. 逐行处理
    for idx, row in df.iterrows():
        question_text = get_question_text(row)
        options_text = get_options_text(row)
        
        # 构造人类提示词
        user_prompt = f"""
        请分析以下试题：
        
        题干：
        {question_text}

        选项：
        {options_text}

        请给出该题的难度和认知层级。"""
        
        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt)
        ]
        
        try:
            # 调用LLM
            response = llm.invoke(messages)
            content = response.content if hasattr(response, 'content') else str(response)
            difficulty, cognitive = parse_llm_response(content)
            
            if difficulty is None:
                print(f"第{idx+1}题：无法解析难度，原始输出：{content[:50]}...",end='  ')
            if cognitive is None:
                print(f"第{idx+1}题：无法解析认知层级，原始输出：{content[:50]}...",end='  ')
                
        except Exception as e:
            print(f"第{idx+1}题调用LLM失败：{e}")
            difficulty, cognitive = None, None
        
        difficulties.append(difficulty)
        cognitive_levels.append(cognitive)
    
    # 5. 将结果添加为新列
    df['难度_预测'] = difficulties
    df['认知层级_预测'] = cognitive_levels
    
    # 6. 保存为新Excel文件
    df.to_excel(output_excel, index=False)
    print(f"处理完成！结果已保存至：{output_excel}")

In [18]:
def get_answer(input_excel: str, output_excel: str):
    """
    主函数2：读取Excel，为每一题调用LLM求答案，输出新Excel。
    """
    # 1. 读取Excel，所有列作为字符串读入，避免科学计数法等干扰
    df = pd.read_excel(input_excel, dtype=str)
    
    # 2. 准备存储结果的列表
    answers=[]
    
    # 3. 系统提示词
    system_prompt = """
        你是一名专业的高考地理命题专家，擅长根据试题的题干解题。
        解题要求：
        阅读理解题干和选项，根据地理学科思维，选择最符合提问的一项。
        请严格按以下格式只输出四个大写字母中的一个，不要输出额外解释、前后缀、空格和格式等：
        A/B/C/D
        """

    # 4. 逐行处理
    for idx, row in df.iterrows():
        question_text = get_question_text(row)
        options_text = get_options_text(row)
        
        # 构造人类提示词
        user_prompt = f"""
        请分析以下试题：
        
        题干：
        {question_text}

        选项：
        {options_text}

        请给出该题的答案。"""
        
        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt)
        ]
        
        try:
            # 调用LLM
            response = llm.invoke(messages)
            content = response.content if hasattr(response, 'content') else str(response)
            
            if content is None:
                print(f"第{idx+1}题：无法求解，原始输出：{content[:50]}...",end='  ')
                
        except Exception as e:
            print(f"第{idx+1}题调用LLM失败：{e}")
            content = None
        
        answers.append(content)
    
    # 5. 将结果添加为新列
    df['答案'] = answers
    
    # 6. 保存为新Excel文件
    df.to_excel(output_excel, index=False)
    print(f"处理完成！结果已保存至：{output_excel}")

In [12]:
# 标注示例
if __name__ == "__main__":
    input_file = "./data/题库.xlsx"   # 请替换为实际文件路径
    output_file = "./data/题库.xlsx"
    labeling(input_file, output_file)

处理完成！结果已保存至：./data/题库.xlsx


In [21]:
llm=llm_qwen

In [19]:
# 求解示例
if __name__ == "__main__":
    input_file = "./data/题库.xlsx"   # 请替换为实际文件路径
    output_file = "./data/题库.xlsx"
    get_answer(input_file, output_file)

处理完成！结果已保存至：./data/题库.xlsx
